In [2]:
!pip install yfinance pandas tabulate -q

import yfinance as yf
import pandas as pd
import numpy as np

# Define Candidate Tickers to Screen
TICKERS = ['VGT', 'PLTR', 'TSM', 'TSLA', 'MSFT', 'MU', 'GOOGL', 'CRWD', 'NVDA', 'ANET', 'SPCX', 'VOO', 'AMZN', 'VRT', 'AVGO', 'DDOG', 'SMH', 'ASML', 'META', 'ORCL', 'BE', 'ZS', 'ETN', 'CEG', 'SNDK', 'AMD', 'ARTY', 'RKLB', 'MDB', 'ARM', 'RDDT', 'GEV', 'AAPL', 'LLY', 'AMAT', 'QQQM', 'LITE', 'PWR', 'SHAZ', 'COHR', 'IGV', 'ASTS', 'APP', 'LRCX', 'QBTS', 'INTC', 'COIN', 'OKLO', 'CDNS', 'SHOP', 'NET', 'ADI', 'TWLO', 'PANW', 'MRVL', 'CRWV', 'NBIS', 'OKTA', 'IREN', 'SOFI']

def evaluate_stock(ticker_symbol):
    stock = yf.Ticker(ticker_symbol)
    info = stock.info

    # Financial Statements
    financials = stock.financials
    balance_sheet = stock.balance_sheet
    cashflow = stock.cashflow

    results = {
        "Ticker": ticker_symbol,
        "Net_Cash": False,
        "Interest_Coverage": None,
        "FCF_Positive": False,
        "FCF_Conversion": None,
        "Dilution_Check": False,
        "Gross_Margin": None,
        "Pass_All_Gates": True,
        "Disqualification_Reasons": []
    }

    # 1. Solvency & Balance Sheet
    total_cash = info.get("totalCash", 0)
    total_debt = info.get("totalDebt", 0)
    operating_income = info.get("operatingMargins", 0) * info.get("totalRevenue", 1)

    if total_cash > total_debt:
        results["Net_Cash"] = True
    else:
        interest_exp = abs(financials.loc["Interest Expense"].iloc[0]) if "Interest Expense" in financials.index else 0
        ebit = financials.loc["Operating Income"].iloc[0] if "Operating Income" in financials.index else 0
        cov = (ebit / interest_exp) if interest_exp > 0 else 999
        results["Interest_Coverage"] = round(cov, 2)
        if cov < 6.0:
            results["Pass_All_Gates"] = False
            results["Disqualification_Reasons"].append(f"Low Interest Coverage ({cov:.1f}x)")

    # 2. Free Cash Flow & Quality
    fcf = info.get("freeCashflow", 0)
    net_income = info.get("netIncomeToCommon", 0)
    results["FCF_Positive"] = fcf > 0
    if not results["FCF_Positive"]:
        results["Pass_All_Gates"] = False
        results["Disqualification_Reasons"].append("Negative FCF")
    elif net_income > 0:
        fcf_conv = fcf / net_income
        results["FCF_Conversion"] = round(fcf_conv, 2)
        if fcf_conv < 0.65:
            results["Pass_All_Gates"] = False
            results["Disqualification_Reasons"].append(f"Low FCF Conversion ({fcf_conv:.2f})")

    # 3. Share Dilution Check (Shares outstanding growth over past 3 periods)
    if "Ordinary Shares Number" in balance_sheet.index:
        shares = balance_sheet.loc["Ordinary Shares Number"]
        if len(shares) >= 3:
            s_now = shares.iloc[0]
            s_past = shares.iloc[2]
            cagr = ((s_now / s_past) ** (1/2) - 1) * 100
            results["Dilution_Check"] = cagr <= 3.0
            if not results["Dilution_Check"]:
                results["Pass_All_Gates"] = False
                results["Disqualification_Reasons"].append(f"High Dilution (CAGR {cagr:.1f}%)")
    else:
        results["Dilution_Check"] = True # Default pass if data point unavailable

    # 4. Gross Margin Check (Unit Economics)
    gm = info.get("grossMargins", 0) * 100
    results["Gross_Margin"] = round(gm, 1)
    if gm < 40.0:
        results["Pass_All_Gates"] = False
        results["Disqualification_Reasons"].append(f"Weak Gross Margin ({gm:.1f}%)")

    # 5. Valuation Sanity (PEG)
    peg = info.get("pegRatio", None)
    if peg and peg > 2.5:
        results["Pass_All_Gates"] = False
        results["Disqualification_Reasons"].append(f"Extreme PEG ({peg:.2f}x)")

    return results

# Run pipeline
records = []
for t in TICKERS:
    try:
        records.append(evaluate_stock(t))
    except Exception as e:
        records.append({
            "Ticker": t, "Pass_All_Gates": False,
            "Disqualification_Reasons": [f"Data parsing error: {str(e)}"]
        })

df = pd.DataFrame(records)

# Display Qualified vs Disqualified
qualified = df[df["Pass_All_Gates"] == True]
disqualified = df[df["Pass_All_Gates"] == False]

print("\n========== QUALIFIED (NON-SPECULATIVE CANDIDATES) ==========")
print(qualified[["Ticker", "Gross_Margin", "FCF_Conversion", "Net_Cash"]].to_string(index=False))

print("\n========== DISQUALIFIED (FILTERED OUT AS SPECULATIVE) ==========")
for _, row in disqualified.iterrows():
    reasons = ", ".join(row["Disqualification_Reasons"]) if row["Disqualification_Reasons"] else "Threshold failed"
    print(f"- {row['Ticker']}: {reasons}")





========== QUALIFIED (NON-SPECULATIVE CANDIDATES) ==========
Ticker  Gross_Margin  FCF_Conversion  Net_Cash
  ANET          63.0            0.96      True
  ASML          52.7            0.79      True
   AMD          55.7            1.37      True
   ARM          97.5            1.28      True
   APP          88.5            0.72     False
  SHOP          47.8            0.82      True
   NET          72.6             NaN      True
   ADI          65.8            1.00     False
  TWLO          48.6            0.83      True
  MRVL          52.2            0.91     False

========== DISQUALIFIED (FILTERED OUT AS SPECULATIVE) ==========
- VGT: Negative FCF, Weak Gross Margin (0.0%)
- PLTR: High Dilution (CAGR 4.3%)
- TSM: Low FCF Conversion (0.33)
- TSLA: High Dilution (CAGR 8.5%), Weak Gross Margin (18.9%), Extreme PEG (4.58x)
- MSFT: Low FCF Conversion (0.12)
- MU: Low FCF Conversion (0.34)
- GOOGL: Low FCF Conversion (0.09)
- CRWD: High Dilution (CAGR 5.1%), Extreme PEG (7.10x)
- NV

'VGT', 'PLTR', 'TSM', 'TSLA', 'MSFT', 'MU', 'GOOGL', 'CRWD', 'NVDA', 'ANET', 'SPCX', 'VOO', 'AMZN', 'VRT', 'AVGO', 'DDOG', 'SMH', 'ASML', 'META', 'ORCL', 'BE', 'ZS', 'ETN', 'CEG', 'SNDK', 'AMD', 'ARTY', 'RKLB', 'MDB', 'ARM', 'RDDT', 'GEV', 'AAPL', 'LLY', 'AMAT', 'QQQM', 'LITE', 'PWR', 'SHAZ', 'COHR', 'IGV', 'ASTS', 'APP', 'LRCX', 'QBTS', 'INTC', 'COIN', 'OKLO', 'CDNS', 'SHOP', 'NET', 'ADI', 'TWLO', 'PANW', 'MRVL', 'CRWV', 'NBIS', 'OKTA', 'IREN', 'SOFI'



APP, GOOGL, META, RDDT, NBIS, TSLA, AMZN, IREN, SOFI, COIN, LLY, SPCX, RKLB, VRT, BE, PWR, ETN, GEV,  VGT, VOO, SMH, ARTY, QQQM, IGV, LITE, ASTS, ANET, SNDK, QBTS, AAPL, SHAZ, COHR, ASML, AMAT, LRCX, TSM, MU, NVDA, AVGO, AMD, ARM, INTC, ADI, MRVL, DDOG, CDNS, SHOP, PLTR, MSFT, CRWD, ORCL, ZS, MDB, NET, TWLO, PANW, CRWV, OKTA, CEG, OKLO


# Yurui's customization


In [3]:
!pip install yfinance pandas tabulate -q

import yfinance as yf
import pandas as pd
import numpy as np

# Define Candidate Tickers to Screen
TICKERS = [ 'CRWD',  'ANET',   'AMZN', 'VRT', 'AVGO', 'DDOG', 'ASML',  'BE', 'ZS', 'ETN', 'CEG', 'SNDK', 'AMD',  'MDB', 'ARM', 'RDDT', 'GEV', 'AAPL', 'LLY', 'AMAT',  'PWR', 'ASTS', 'APP', 'LRCX',  'INTC', 'COIN',  'CDNS', 'SHOP', 'NET', 'ADI', 'TWLO', 'PANW', 'MRVL', 'CRWV', 'NBIS', 'OKTA', 'IREN']

def evaluate_stock(ticker_symbol):
    stock = yf.Ticker(ticker_symbol)
    info = stock.info

    # Financial Statements
    financials = stock.financials
    balance_sheet = stock.balance_sheet
    cashflow = stock.cashflow

    results = {
        "Ticker": ticker_symbol,
        "Net_Cash": False,
        "Interest_Coverage": None,
        "FCF_Positive": False,
        "FCF_Conversion": None,
        "Dilution_Check": False,
        "Gross_Margin": None,
        "Pass_All_Gates": True,
        "Disqualification_Reasons": []
    }

    # 1. Solvency & Balance Sheet
    total_cash = info.get("totalCash", 0)
    total_debt = info.get("totalDebt", 0)
    operating_income = info.get("operatingMargins", 0) * info.get("totalRevenue", 1)

    if total_cash > total_debt:
        results["Net_Cash"] = True
    else:
        interest_exp = abs(financials.loc["Interest Expense"].iloc[0]) if "Interest Expense" in financials.index else 0
        ebit = financials.loc["Operating Income"].iloc[0] if "Operating Income" in financials.index else 0
        cov = (ebit / interest_exp) if interest_exp > 0 else 999
        results["Interest_Coverage"] = round(cov, 2)
        if cov < 6.0:
            results["Pass_All_Gates"] = False
            results["Disqualification_Reasons"].append(f"Low Interest Coverage ({cov:.1f}x)")

    # 2. Free Cash Flow & Quality
    fcf = info.get("freeCashflow", 0)
    net_income = info.get("netIncomeToCommon", 0)
    results["FCF_Positive"] = fcf > 0
    if not results["FCF_Positive"]:
        results["Pass_All_Gates"] = False
        results["Disqualification_Reasons"].append("Negative FCF")
    elif net_income > 0:
        fcf_conv = fcf / net_income
        results["FCF_Conversion"] = round(fcf_conv, 2)

        # change from 0.65 to 0.3
        if fcf_conv < 0.3:


            results["Pass_All_Gates"] = False
            results["Disqualification_Reasons"].append(f"Low FCF Conversion ({fcf_conv:.2f})")

    # 3. Share Dilution Check (Shares outstanding growth over past 3 periods)
    if "Ordinary Shares Number" in balance_sheet.index:
        shares = balance_sheet.loc["Ordinary Shares Number"]
        if len(shares) >= 3:
            s_now = shares.iloc[0]
            s_past = shares.iloc[2]
            cagr = ((s_now / s_past) ** (1/2) - 1) * 100
            results["Dilution_Check"] = cagr <= 3.0
            if not results["Dilution_Check"]:
                results["Pass_All_Gates"] = False
                results["Disqualification_Reasons"].append(f"High Dilution (CAGR {cagr:.1f}%)")
    else:
        results["Dilution_Check"] = True # Default pass if data point unavailable

    # 4. Gross Margin Check (Unit Economics)
    gm = info.get("grossMargins", 0) * 100
    results["Gross_Margin"] = round(gm, 1)
    if gm < 40.0:
        results["Pass_All_Gates"] = False
        results["Disqualification_Reasons"].append(f"Weak Gross Margin ({gm:.1f}%)")

    # 5. Valuation Sanity (PEG)
    peg = info.get("pegRatio", None)
    if peg and peg > 2.5:
        results["Pass_All_Gates"] = False
        results["Disqualification_Reasons"].append(f"Extreme PEG ({peg:.2f}x)")

    return results

# Run pipeline
records = []
for t in TICKERS:
    try:
        records.append(evaluate_stock(t))
    except Exception as e:
        records.append({
            "Ticker": t, "Pass_All_Gates": False,
            "Disqualification_Reasons": [f"Data parsing error: {str(e)}"]
        })

df = pd.DataFrame(records)

# Display Qualified vs Disqualified
qualified = df[df["Pass_All_Gates"] == True]
disqualified = df[df["Pass_All_Gates"] == False]

print("\n========== QUALIFIED (NON-SPECULATIVE CANDIDATES) ==========")
print(qualified[["Ticker", "Gross_Margin", "FCF_Conversion", "Net_Cash"]].to_string(index=False))

print("\n========== DISQUALIFIED (FILTERED OUT AS SPECULATIVE) ==========")
for _, row in disqualified.iterrows():
    reasons = ", ".join(row["Disqualification_Reasons"]) if row["Disqualification_Reasons"] else "Threshold failed"
    print(f"- {row['Ticker']}: {reasons}")





========== QUALIFIED (NON-SPECULATIVE CANDIDATES) ==========
Ticker  Gross_Margin  FCF_Conversion  Net_Cash
  ANET          63.0            0.96      True
  ASML          52.7            0.79      True
   AMD          55.7            1.37      True
   ARM          97.5            1.28      True
   LLY          83.4            0.41     False
  AMAT          49.4            0.33      True
   APP          88.5            0.72     False
  LRCX          50.5            0.43      True
  SHOP          47.8            0.82      True
   NET          72.6             NaN      True
   ADI          65.8            1.00     False
  TWLO          48.6            0.83      True
  MRVL          52.2            0.91     False

========== DISQUALIFIED (FILTERED OUT AS SPECULATIVE) ==========
- CRWD: High Dilution (CAGR 5.1%), Extreme PEG (7.10x)
- AMZN: Low FCF Conversion (0.02)
- VRT: Weak Gross Margin (38.0%)
- AVGO: High Dilution (CAGR 7.0%)
- DDOG: High Dilution (CAGR 3.2%)
- BE: Low Interest Cover